In [1]:
# %% Cell 1 - Setup
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# %% Cell 2 - Markov chain functions

block_size = 512
batch_size = 16
learning_rate = 0.001
weight_decay = 1e-3
dropout = 0.0
vocabulary_size = 2
eval_iters = 200
eval_interval = 500

def generate_transition_matrix(order, seed):
    if seed is not None:
        torch.manual_seed(seed)
    n_states = 2 ** order
    matrix = torch.rand((n_states, 2))
    row_sums = matrix.sum(dim=1, keepdim=True)
    matrix = matrix / row_sums
    return matrix

def generate_sequence(transition_matrix, order, length, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    sequence = torch.zeros(length, dtype=torch.long)
    for i in range(order):
        sequence[i] = torch.randint(0, 2, (1,))
    for i in range(order, length):
        state_idx = 0
        for j in range(order):
            state_idx = state_idx * 2 + sequence[i - order + j]
        probs = transition_matrix[int(state_idx)]
        sequence[i] = torch.bernoulli(probs[1])
    return sequence.tolist()

def estimate_matrix(sequence, order):
    num_states = 2 ** order
    counts = torch.zeros((num_states, 2))
    for i in range(len(sequence) - order):
        state = sequence[i:i + order]
        state_idx = int(''.join(map(str, state)), 2)
        next_bit = sequence[i + order]
        counts[state_idx, next_bit] += 1
    row_sums = counts.sum(dim=1, keepdim=True)
    row_sums[row_sums == 0] = 2
    return counts / row_sums

def generate_training_data(transition_matrix, order, sequence_length, num_sequences):
    sequences = []
    for i in range(num_sequences):
        sequence = generate_sequence(transition_matrix, order, sequence_length, i)
        sequences.append(torch.tensor(sequence, dtype=torch.long))
    return sequences

def get_batch(sequences, split='train'):
    n = int(0.9 * len(sequences))
    data = sequences[:n] if split == 'train' else sequences[n:]
    while True:
        seq_indices = torch.randint(len(data), (batch_size,))
        x_batch, y_batch = [], []
        for idx in seq_indices:
            sequence = data[idx]
            if len(sequence) >= block_size + 1:
                start_idx = torch.randint(0, len(sequence) - block_size, (1,))
                x_batch.append(sequence[start_idx:start_idx + block_size])
                y_batch.append(sequence[start_idx + 1:start_idx + block_size + 1])
        if len(x_batch) > 0:
            return torch.stack(x_batch).to(device), torch.stack(y_batch).to(device)

@torch.no_grad()
def estimate_loss(model, sequences):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(sequences, split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def kl_divergence(chain1, chain2, epsilon=1e-10):
    defined_states = torch.sum(chain2, dim=1) > 0
    p = chain1[defined_states] + epsilon
    q = chain2[defined_states] + epsilon
    p = p / p.sum(dim=1, keepdim=True)
    q = q / q.sum(dim=1, keepdim=True)
    return torch.sum(p * torch.log(p / q)).item()

def mean_error(chain1, chain2):
    defined_states = torch.sum(chain2, dim=1) > 0
    return torch.mean(torch.abs(chain1[defined_states] - chain2[defined_states])).item()

In [4]:
# %% Cell 3 - Transformer model

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BinaryTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocabulary_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocabulary_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("Model defined.")

Model defined.


In [5]:
# %% Cell 4 - Training function

def train_model(order, model_id, num_iterations, seed, sequence_length, num_sequences, save_dir):
    model = BinaryTransformer().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    transition_matrix = generate_transition_matrix(order, seed)
    sequences = generate_training_data(transition_matrix, order, sequence_length, num_sequences)
    
    n_states = 2 ** order
    entropy_rate = 0
    for s in range(n_states):
        for b in range(2):
            p = transition_matrix[s, b].item()
            if p > 0:
                entropy_rate -= (1/n_states) * p * np.log(p)

    history = {'train': [], 'val': [], 'iterations': []}

    for iter in range(num_iterations):
        xb, yb = get_batch(sequences, 'train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if iter % eval_interval == 0 or iter == num_iterations - 1:
            losses = estimate_loss(model, sequences)
            history['train'].append(losses['train'].item())
            history['val'].append(losses['val'].item())
            history['iterations'].append(iter)
            print(f"  iter {iter}: train {losses['train']:.4f}, val {losses['val']:.4f}, entropy rate {entropy_rate:.4f}")

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'transition_matrix': transition_matrix.tolist(),
        'history': history,
        'config': {'order': order, 'n_head': n_head, 'n_layer': n_layer,
                   'n_embd': n_embd, 'iterations': num_iterations,
                   'seq_length': sequence_length, 'num_sequences': num_sequences,
                   'seed': seed, 'entropy_rate': entropy_rate},
    }, save_path / f'model_o{order}_m{model_id}.pt')

    return {
        'order': order, 'model_id': model_id, 'seed': seed,
        'transition_matrix': transition_matrix,
        'history': history, 'entropy_rate': entropy_rate,
    }

print("Training function defined.")

Training function defined.


In [6]:
# %% Cell 5 - Experiment 2: Scaling model capacity
# Keep Moritz's data settings, increase model size
# Use the SCALED data from exp1 (more data can't hurt)

results_dir = os.path.expanduser('../results/model_scaling')
models_dir = os.path.expanduser('../models/exp2')

seeds = list(range(42, 52))  # 10 seeds

configs = {
    1: {
        'moritz':  {'n_head': 1, 'n_layer': 2, 'n_embd': 16,  'iters': 5000,  'seq': 1000,  'nseq': 500},
        'medium':  {'n_head': 2, 'n_layer': 4, 'n_embd': 64,  'iters': 10000, 'seq': 1000,  'nseq': 500},
        'large':   {'n_head': 4, 'n_layer': 6, 'n_embd': 128, 'iters': 10000, 'seq': 1000,  'nseq': 500},
    },
    2: {
        'moritz':  {'n_head': 1, 'n_layer': 2, 'n_embd': 64,  'iters': 5000,  'seq': 2000,  'nseq': 1000},
        'medium':  {'n_head': 2, 'n_layer': 4, 'n_embd': 128, 'iters': 10000, 'seq': 2000,  'nseq': 1000},
        'large':   {'n_head': 4, 'n_layer': 6, 'n_embd': 256, 'iters': 15000, 'seq': 2000,  'nseq': 1000},
    },
    3: {
        'moritz':  {'n_head': 1, 'n_layer': 3, 'n_embd': 64,  'iters': 10000, 'seq': 2000,  'nseq': 1000},
        'medium':  {'n_head': 2, 'n_layer': 4, 'n_embd': 128, 'iters': 15000, 'seq': 2000,  'nseq': 1000},
        'large':   {'n_head': 4, 'n_layer': 6, 'n_embd': 256, 'iters': 20000, 'seq': 2000,  'nseq': 1000},
    },
    4: {
        'moritz':  {'n_head': 1, 'n_layer': 3, 'n_embd': 64,  'iters': 10000, 'seq': 4000,  'nseq': 1000},
        'medium':  {'n_head': 2, 'n_layer': 4, 'n_embd': 128, 'iters': 15000, 'seq': 4000,  'nseq': 1000},
        'large':   {'n_head': 4, 'n_layer': 6, 'n_embd': 256, 'iters': 20000, 'seq': 4000,  'nseq': 1000},
    },
    5: {
        'moritz':  {'n_head': 1, 'n_layer': 4, 'n_embd': 64,  'iters': 15000, 'seq': 8000,  'nseq': 2000},
        'medium':  {'n_head': 2, 'n_layer': 6, 'n_embd': 128, 'iters': 20000, 'seq': 8000,  'nseq': 2000},
        'large':   {'n_head': 4, 'n_layer': 8, 'n_embd': 256, 'iters': 25000, 'seq': 8000,  'nseq': 2000},
    },
    6: {
        'moritz':  {'n_head': 1, 'n_layer': 4, 'n_embd': 64,  'iters': 15000, 'seq': 16000, 'nseq': 2000},
        'medium':  {'n_head': 2, 'n_layer': 6, 'n_embd': 128, 'iters': 20000, 'seq': 16000, 'nseq': 2000},
        'large':   {'n_head': 4, 'n_layer': 8, 'n_embd': 256, 'iters': 25000, 'seq': 16000, 'nseq': 2000},
    },
}

# Print config summary
print("Experiment 2: Model Capacity Scaling")
print(f"{'Order':<7} {'Size':<8} {'Heads':<7} {'Layers':<8} {'Embd':<7} {'Iters':<8} {'Params (approx)'}")
print("-" * 70)
for order in configs:
    for size in ['moritz', 'medium', 'large']:
        cfg = configs[order][size]
        # Rough param count
        params = cfg['n_embd'] * 2 + cfg['n_embd'] * block_size  # embeddings
        params += cfg['n_layer'] * (4 * cfg['n_embd']**2 + 8 * cfg['n_embd']**2)  # attention + ff
        print(f"{order:<7} {size:<8} {cfg['n_head']:<7} {cfg['n_layer']:<8} {cfg['n_embd']:<7} {cfg['iters']:<8} ~{params:,}")

Experiment 2: Model Capacity Scaling
Order   Size     Heads   Layers   Embd    Iters    Params (approx)
----------------------------------------------------------------------
1       moritz   1       2        16      5000     ~14,368
1       medium   2       4        64      10000    ~229,504
1       large    4       6        128     10000    ~1,245,440
2       moritz   1       2        64      5000     ~131,200
2       medium   2       4        128     10000    ~852,224
2       large    4       6        256     15000    ~4,850,176
3       moritz   1       3        64      10000    ~180,352
3       medium   2       4        128     15000    ~852,224
3       large    4       6        256     20000    ~4,850,176
4       moritz   1       3        64      10000    ~180,352
4       medium   2       4        128     15000    ~852,224
4       large    4       6        256     20000    ~4,850,176
5       moritz   1       4        64      15000    ~229,504
5       medium   2       6        128 

In [ ]:
# %% Cell 6 - Train all model sizes

all_results = {}

for order, order_configs in configs.items():
    all_results[order] = {}
    
    for size in ['moritz', 'medium', 'large']:
        cfg = order_configs[size]
        n_head = cfg['n_head']
        n_layer = cfg['n_layer']
        n_embd = cfg['n_embd']
        
        all_results[order][size] = []
        
        print(f"\n{'='*60}")
        print(f"ORDER {order} — {size.upper()} ({cfg['n_head']}h, {cfg['n_layer']}L, {cfg['n_embd']}d, {cfg['iters']} iters)")
        print(f"{'='*60}")
        
        for i, seed in enumerate(seeds):
            print(f"\n  Model {i+1}/10 (seed={seed})")
            result = train_model(
                order=order, model_id=i,
                num_iterations=cfg['iters'], seed=seed,
                sequence_length=cfg['seq'],
                num_sequences=cfg['nseq'],
                save_dir=f"{models_dir}/{size}/order{order}"
            )
            all_results[order][size].append(result)

print("\n\nALL TRAINING COMPLETE!")


ORDER 1 — MORITZ (1h, 2L, 16d, 5000 iters)

  Model 1/10 (seed=42)
  iter 0: train 0.7177, val 0.7191, entropy rate 0.6454
  iter 500: train 0.6316, val 0.6344, entropy rate 0.6454
  iter 1000: train 0.6312, val 0.6343, entropy rate 0.6454
  iter 1500: train 0.6316, val 0.6349, entropy rate 0.6454
  iter 2000: train 0.6312, val 0.6338, entropy rate 0.6454
  iter 2500: train 0.6315, val 0.6341, entropy rate 0.6454
  iter 3000: train 0.6319, val 0.6338, entropy rate 0.6454
  iter 3500: train 0.6313, val 0.6340, entropy rate 0.6454
  iter 4000: train 0.6317, val 0.6339, entropy rate 0.6454
  iter 4500: train 0.6315, val 0.6344, entropy rate 0.6454
